# EDA on credit labels and silver data

Moved out of `main.py` so the pipeline only builds tables. Run `python main.py` first so `datamart/` exists.

In [ ]:
import os
import glob
from datetime import datetime

import matplotlib.pyplot as plt
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col

spark = pyspark.sql.SparkSession.builder.appName("eda").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

silver_lms_daily_directory = "datamart/silver/lms_loan_daily/"
silver_financials_daily_directory = "datamart/silver/features_financials/"

## Bad rate by month on book (supports the 30 DPD / MOB 6 label definition)

In [ ]:
# set dpd label definition
dpd = 30

# Read all silver LMS partitions into a single DataFrame
folder_path = silver_lms_daily_directory
files_list = [folder_path + os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)

# filter only completed loans
df = df.filter(col("loan_start_date") < datetime.strptime("2024-01-01", "%Y-%m-%d"))

# create dpd flag if more than dpd
df = df.withColumn("dpd_flag", F.when(col("dpd") >= dpd, 1).otherwise(0))

# actual bads
actual_bads_df = df.filter(col("installment_num") == 10)

# visualise bad rate
pdf = df.toPandas()
grouped = pdf.groupby('mob')['dpd_flag'].mean().sort_index()

grouped.plot(kind='line', marker='o')
plt.title('DPD: ' + str(dpd))
plt.xlabel('mob')
plt.ylabel('bad rate')
plt.grid(True)
plt.savefig('datamart/bad_rate_by_mob.png')
plt.show()

## Check the silver financials cleaning

No negatives, caps respected, and the `__10000__` placeholder removed.

In [ ]:
fin = spark.read.parquet(*glob.glob(silver_financials_daily_directory + '*'))

checks = ["Num_Bank_Accounts", "Num_Credit_Card", "Interest_Rate", "Num_of_Loan",
          "Num_of_Delayed_Payment", "Num_Credit_Inquiries", "Delay_from_due_date"]
fin.select([F.min(c).alias(c + "_min") for c in checks] + [F.max(c).alias(c + "_max") for c in checks]).show(vertical=True)

print("Amount_invested_monthly == 10000:", fin.filter(col("Amount_invested_monthly") == 10000).count())
fin.select([F.sum(col(c).isNull().cast("int")).alias(c + "_nulls") for c in checks + ["Amount_invested_monthly"]]).show(vertical=True)